In [1]:
import os

os.environ[
    "TF_USE_LEGACY_KERAS"
] = "1"

In [2]:
!pip install -q tensorflow-recommenders

In [3]:
import pandas as pd
import pickle

import tensorflow as tf
import tensorflow_recommenders as tfrs

2026-05-14 08:33:45.334078: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778747625.359452     119 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778747625.366761     119 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778747625.386315     119 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778747625.386349     119 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778747625.386352     119 computation_placer.cc:177] computation placer alr

In [4]:
print("TensorFlow:", tf.__version__)
print("TFRS:", tfrs.__version__)

TensorFlow: 2.19.0
TFRS: v0.7.7


Load the training data

In [5]:
training_df = pd.read_parquet(
    "/kaggle/input/datasets/selinparmar/train-ds/training_dataset.parquet"
)

print(training_df.shape)
training_df.head()

(1000000, 16)


,customer_id,article_id,age,purchase_frequency,days_since_last_purchase,avg_spend,avg_days_between_purchases,repeat_purchase_ratio,favorite_product_type,product_type_name,season,sales_channel_id,purchase_count,month,day_of_week,interaction_label
0,0bae744d492a1ee45460ef798a977b33ffa09f14f92cf6...,632307002,23.0,8.0,295.0,0.030057,13.857143,0.000000,Blazer,Top,Autumn,1,1846.0,10,Thursday,1
1,a076c84a1df6ecdb4f21bd6bac49ac5e6704aa48340b7c...,688728003,44.0,126.0,11.0,0.024303,5.560000,0.067797,Trousers,Underwear bottom,Autumn,2,3013.0,11,Sunday,1
2,e1262cea5799e7f866272c5b54aeab5641e62926a4bffe...,524529007,48.0,18.0,457.0,0.018544,0.000000,0.000000,Vest top,Vest top,Summer,1,1234.0,6,Sunday,1
3,17c6855702b6e50f732bee25ef6789927dc3cb61161dfc...,800529001,32.0,219.0,6.0,0.026946,3.041284,0.038095,Dress,Trousers,Summer,2,190.0,1,Thursday,1
4,1342bda603bf7ae79af2595951f069db1b50a3cc5dbcc4...,758129001,28.0,97.0,17.0,0.033973,7.281250,0.054348,Dress,Dress,Winter,2,606.0,7,Tuesday,1


In [7]:
with open(
    "/kaggle/input/datasets/selinparmar/feature-vocab/feature_vocab.pkl",
    "rb"
) as file:

    feature_vocab = pickle.load(file)

print(feature_vocab.keys())

dict_keys(['user_vocab', 'item_vocab', 'product_type_vocab', 'favorite_product_vocab', 'season_vocab', 'day_vocab'])


In [8]:
training_df[
    'customer_id'
] = (
    training_df[
        'customer_id'
    ].astype(str)
)

training_df[
    'article_id'
] = (
    training_df[
        'article_id'
    ].astype(str)
)

Create tensorflow dataset

In [9]:
tf_dataset = tf.data.Dataset.from_tensor_slices(
    {
        key:
        training_df[key].values

        for key in training_df.columns
    }
)

print(tf_dataset)

2026-05-14 08:42:00.849276: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


<_TensorSliceDataset element_spec={'customer_id': TensorSpec(shape=(), dtype=tf.string, name=None), 'article_id': TensorSpec(shape=(), dtype=tf.string, name=None), 'age': TensorSpec(shape=(), dtype=tf.float64, name=None), 'purchase_frequency': TensorSpec(shape=(), dtype=tf.float64, name=None), 'days_since_last_purchase': TensorSpec(shape=(), dtype=tf.float64, name=None), 'avg_spend': TensorSpec(shape=(), dtype=tf.float64, name=None), 'avg_days_between_purchases': TensorSpec(shape=(), dtype=tf.float64, name=None), 'repeat_purchase_ratio': TensorSpec(shape=(), dtype=tf.float64, name=None), 'favorite_product_type': TensorSpec(shape=(), dtype=tf.string, name=None), 'product_type_name': TensorSpec(shape=(), dtype=tf.string, name=None), 'season': TensorSpec(shape=(), dtype=tf.string, name=None), 'sales_channel_id': TensorSpec(shape=(), dtype=tf.int64, name=None), 'purchase_count': TensorSpec(shape=(), dtype=tf.float64, name=None), 'month': TensorSpec(shape=(), dtype=tf.int32, name=None), 'da

In [11]:
dataset_size = len(training_df)

train_size = int(
    0.8 * dataset_size
)

train = (
    tf_dataset
    .take(train_size)
)

test = (
    tf_dataset
    .skip(train_size)
)

In [12]:
BATCH_SIZE = 4096

train = (
    train
    .shuffle(100000)
    .batch(BATCH_SIZE)
    .cache()
)

test = (
    test
    .batch(BATCH_SIZE)
    .cache()
)

In [13]:
for batch in train.take(1):

    print(batch.keys())

dict_keys(['customer_id', 'article_id', 'age', 'purchase_frequency', 'days_since_last_purchase', 'avg_spend', 'avg_days_between_purchases', 'repeat_purchase_ratio', 'favorite_product_type', 'product_type_name', 'season', 'sales_channel_id', 'purchase_count', 'month', 'day_of_week', 'interaction_label'])


In [14]:
for batch in train.take(1):

    print(
        batch['customer_id'][:5]
    )

    print(
        batch['article_id'][:5]
    )

    print(
        batch[
            'favorite_product_type'
        ][:5]
    )

tf.Tensor(
[b'dbc72c56bdac1f2b7f067bb2aea67ac4582a439bdfa8e684e104ef796401d049'
 b'3ab4f6998a6ea572645a0b057294fe4e3d4f6f3a96793d66a2496d0dd0d558b8'
 b'3247341a7d5ff42c06594e8c4f84d726bcf632346a0ca5cd0f1a66b4521bee00'
 b'789d417b9c6fdd323b6740f2ccf7ad4695c37ce9bf711f3ad466984423da66bd'
 b'ba0794162397b835b34b49e61a2bae0e6fb84fa508a24cacf9357e36fd1af7cc'], shape=(5,), dtype=string)
tf.Tensor([b'905895001' b'859247001' b'696738001' b'835008011' b'805011001'], shape=(5,), dtype=string)
tf.Tensor([b'Blouse' b'Blouse' b'T-shirt' b'Swimsuit' b'Vest top'], shape=(5,), dtype=string)
